# 04 -- Stage 4: Add Synthetic Playground Data

Batch-generates synthetic manipulations over real images using the
same `services/playground/controller.py` the live app uses, appends
them to the training manifest, and retrains.

In [ ]:
import sys, os
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
else:
    print("Not in Colab -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import cv2, random, pandas as pd
from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.playground.controller import generate

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
images_dir = dataset_root / settings.data.images_dir
synthetic_dir = resolve_path(settings.data.synthetic_dir)
synthetic_dir.mkdir(parents=True, exist_ok=True)

manipulation_types = ["copy_move", "splicing", "compression", "blur", "beautify"]
rows = []

for image_path in images_dir.glob("*.jpg"):
    image = cv2.imread(str(image_path))
    if image is None:
        continue
    manipulation_type = random.choice(manipulation_types)
    result = generate(manipulation_type, image)

    out_name = f"{image_path.stem}_{manipulation_type}.jpg"
    out_path = synthetic_dir / out_name
    cv2.imwrite(str(out_path), result.image)
    rows.append({"filename": str(out_path), "label": result.label, "type": result.manipulation_type})

synthetic_manifest = resolve_path("data/synthetic_labels.csv")
pd.DataFrame(rows).to_csv(synthetic_manifest, index=False)
print("Generated", len(rows), "synthetic samples ->", synthetic_manifest)

Concatenate `data/synthetic_labels.csv` onto your Stage 3 merged
manifest and rerun training from the latest checkpoint.